# Multidimensional Similarity Analysis, Trait Signaling and Spillover Effects

This Colab notebook performs a leakage-safe participant-level clustered split (`ResponseId`) for long-format repeated-measures ad data, then produces:

- **Table 7**: Advertisement Personality Profile Matrix (5D vectors)
- **Table 8**: Similarity Metrics Comparison + click correlations + regression comparison
- **Table 9**: Cross-Group Validation / vector stability
- **Appendix figures**: PCA 3D projection, radar chart(s), cosine angle visualization

All outputs are saved to `/content/outputs`.

In [ ]:
# Colab data loading cell: upload or Google Drive mount
import os

USE_DRIVE = False  # set True to load from Google Drive path below
DRIVE_FILE_PATH = '/content/drive/MyDrive/longforpython.xlsx'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    data_path = DRIVE_FILE_PATH
else:
    from google.colab import files
    uploaded = files.upload()
    if 'longforpython.xlsx' in uploaded:
        data_path = '/content/longforpython.xlsx'
    elif len(uploaded) == 1:
        # fallback if user uploads file under different name
        only_name = list(uploaded.keys())[0]
        data_path = f'/content/{only_name}'
        print(f"[INFO] Uploaded file name is '{only_name}'. Using it as input.")
    else:
        raise FileNotFoundError(
            "Please upload longforpython.xlsx (or a single .xlsx file)."
        )

print('Data path:', data_path)

In [ ]:
# Imports and global configuration
import os
import warnings
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

import statsmodels.api as sm

warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

OUTPUT_DIR = '/content/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Notebook parameters
SIGNATURE_OUTCOME = 'PERSUASIVE'  # choose from: PERSUASIVE, RELEVANCE, CLICKTHRU
MIN_TRAIN_N_PER_AD = 30
COLLAPSE_TO_HIGH_ONLY = False  # True = keep only *_H_* ad conditions for tables/figures
RADAR_MODE = 'combined'  # 'combined' or 'per_condition'

print('Configuration:')
print('  SIGNATURE_OUTCOME =', SIGNATURE_OUTCOME)
print('  MIN_TRAIN_N_PER_AD =', MIN_TRAIN_N_PER_AD)
print('  COLLAPSE_TO_HIGH_ONLY =', COLLAPSE_TO_HIGH_ONLY)
print('  RADAR_MODE =', RADAR_MODE)
print('  OUTPUT_DIR =', OUTPUT_DIR)

## 1) Load data and validate required columns

The notebook automatically checks that critical columns exist and are usable.

In [ ]:
# Load data
raw_df = pd.read_excel(data_path)

# Replace common null-like markers
raw_df = raw_df.replace(['#NULL!', 'NULL', 'null', 'NaN', 'nan', ''], np.nan)

required_columns = {
    'id': 'ResponseId',
    'ad': 'AdCond',
    'traits': ['ZOPEN_Score', 'ZCONSC_Score', 'ZEXTRA_Score', 'ZAGREE_Score', 'ZNEURO_Score'],
    'outcomes': ['RELEVANCE', 'PERSUASIVE', 'CLICKTHRU']
}

missing = []
for c in [required_columns['id'], required_columns['ad'], *required_columns['traits'], *required_columns['outcomes']]:
    if c not in raw_df.columns:
        missing.append(c)

if missing:
    raise ValueError(
        'Missing required column(s): ' + ', '.join(missing) +
        '. Please ensure the input file includes all required columns.'
    )

# Keep only required columns for core modeling; retain full frame if needed
critical_cols = [required_columns['id'], required_columns['ad'], *required_columns['traits'], *required_columns['outcomes']]
df = raw_df[critical_cols].copy()

# Numeric conversion
for c in required_columns['traits'] + required_columns['outcomes']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# Basic diagnostics before dropping NA
print('Total raw rows:', len(df))
print('Unique participants (raw):', df[required_columns['id']].nunique())
print('
Counts per AdCond (raw):')
print(df[required_columns['ad']].value_counts(dropna=False).head(50))

# Variance checks for trait z-score columns
zero_var_cols = [c for c in required_columns['traits'] if df[c].dropna().var() == 0 or np.isnan(df[c].dropna().var())]
if zero_var_cols:
    raise ValueError(
        'Trait Z-score column(s) with zero/undefined variance: ' + ', '.join(zero_var_cols) +
        '. Cannot estimate vectors reliably.'
    )

# Drop missing critical values
before_n = len(df)
df = df.dropna(subset=critical_cols)
after_n = len(df)
print(f"
Dropped rows with missing critical values: {before_n - after_n}")
print('Rows retained:', after_n)
print('Unique participants retained:', df[required_columns['id']].nunique())

if after_n == 0:
    raise ValueError('No rows remain after dropping missing critical values.')

if COLLAPSE_TO_HIGH_ONLY:
    before = len(df)
    df = df[df[required_columns['ad']].astype(str).str.contains('_H_')].copy()
    print(f"Applied HIGH-only filter (_H_): {before} -> {len(df)} rows")
    if len(df) == 0:
        raise ValueError('No rows after applying HIGH-only ad filter.')

## 2) Clustered split by participant (ResponseId)

This prevents leakage across repeated rows from the same participant.

In [ ]:
ID_COL = required_columns['id']
AD_COL = required_columns['ad']
TRAIT_COLS = required_columns['traits']
OUTCOME_COLS = required_columns['outcomes']

participant_ids = df[ID_COL].dropna().unique()
train_ids, test_ids = train_test_split(participant_ids, test_size=0.5, random_state=RANDOM_SEED)

train_df = df[df[ID_COL].isin(train_ids)].copy()
test_df = df[df[ID_COL].isin(test_ids)].copy()

# Leakage check
overlap = set(train_df[ID_COL].unique()).intersection(set(test_df[ID_COL].unique()))
if overlap:
    raise RuntimeError('Leakage detected: some participants appear in both TRAIN and TEST.')

print('Train participants:', len(train_ids))
print('Test participants :', len(test_ids))
print('Train rows:', len(train_df))
print('Test rows :', len(test_df))

## 3) Build TRAIN-derived ad signature vectors (Table 7 basis)

For each `AdCond`, estimate:

\[
	ext{Outcome} \sim ZOPEN + ZCONSC + ZEXTRA + ZAGREE + ZNEURO
\]

Using OLS on TRAIN only. Coefficients on z-scored predictors are used as comparable beta-like signature components.

In [ ]:
def fit_ad_signature_vectors(train_data, ad_col, trait_cols, outcome_col, min_n=30):
    records = []
    skipped = []

    for ad, sub in train_data.groupby(ad_col):
        n = len(sub)
        if n < min_n:
            skipped.append((ad, n, 'insufficient_n'))
            continue

        y = sub[outcome_col]
        X = sub[trait_cols]

        # Additional variance guard within-condition
        if (X.var() == 0).any():
            skipped.append((ad, n, 'zero_trait_variance_within_ad'))
            continue
        if y.var() == 0 or np.isnan(y.var()):
            skipped.append((ad, n, 'zero_outcome_variance_within_ad'))
            continue

        X_const = sm.add_constant(X)
        model = sm.OLS(y, X_const).fit()

        vec = model.params[trait_cols].copy()
        norm = np.linalg.norm(vec.values)
        if norm == 0:
            skipped.append((ad, n, 'zero_vector_norm'))
            continue
        vec_unit = vec / norm

        rec = {
            ad_col: ad,
            'n_train': n,
            'vector_norm': norm,
        }
        for t in trait_cols:
            rec[t] = vec[t]
            rec[f'{t}_unit'] = vec_unit[t]
        records.append(rec)

    vectors = pd.DataFrame(records).sort_values(ad_col).reset_index(drop=True)
    skipped_df = pd.DataFrame(skipped, columns=[ad_col, 'n_train', 'reason']) if skipped else pd.DataFrame(columns=[ad_col, 'n_train', 'reason'])
    return vectors, skipped_df

vector_df, skipped_ads = fit_ad_signature_vectors(
    train_data=train_df,
    ad_col=AD_COL,
    trait_cols=TRAIT_COLS,
    outcome_col=SIGNATURE_OUTCOME,
    min_n=MIN_TRAIN_N_PER_AD
)

if vector_df.empty:
    raise ValueError(
        f'No ad conditions had enough TRAIN data to estimate vectors (min_n={MIN_TRAIN_N_PER_AD}).'
    )

print('Estimated vectors for ad conditions:', len(vector_df))
if len(skipped_ads) > 0:
    print('
Skipped ad conditions:')
    print(skipped_ads)

# Table 7 export
table7 = vector_df.copy()

table7.to_csv(os.path.join(OUTPUT_DIR, 'table7.csv'), index=False)
print('Saved:', os.path.join(OUTPUT_DIR, 'table7.csv'))

display(table7.head(20))

## 4) Apply TRAIN vectors to TEST rows and compute similarity metrics

In [ ]:
# Merge ad vectors onto TEST rows
merge_cols = [AD_COL] + TRAIT_COLS + [f'{c}_unit' for c in TRAIT_COLS] + ['vector_norm']
vec_map = vector_df[merge_cols].copy()

# Rename ad-vector component columns to avoid collision with participant trait columns
rename_map = {c: f'{c}_ad' for c in TRAIT_COLS}
rename_map.update({f'{c}_unit': f'{c}_unit_ad' for c in TRAIT_COLS})
vec_map = vec_map.rename(columns=rename_map)

test_eval = test_df.merge(vec_map, on=AD_COL, how='inner')

if test_eval.empty:
    raise ValueError('No TEST rows matched ad conditions with estimable TRAIN vectors.')

# Participant vectors
P = test_eval[TRAIT_COLS].values.astype(float)
# Ad vectors (raw beta-like coefficients)
V = test_eval[[f'{c}_ad' for c in TRAIT_COLS]].values.astype(float)

# Metrics
p_norm = np.linalg.norm(P, axis=1)
v_norm = np.linalg.norm(V, axis=1)
dot = np.sum(P * V, axis=1)

cosine = dot / (p_norm * v_norm)
cosine = np.where(np.isfinite(cosine), cosine, np.nan)
euclidean = np.linalg.norm(P - V, axis=1)

test_eval['CosineSimilarity'] = cosine
test_eval['EuclideanDistance'] = euclidean

# Drop inf/nan for downstream
test_eval = test_eval.replace([np.inf, -np.inf], np.nan)

print('TEST rows with similarity metrics:', len(test_eval))
display(test_eval[[ID_COL, AD_COL, 'CosineSimilarity', 'EuclideanDistance', 'CLICKTHRU']].head())

## 5) Table 8: Similarity metrics comparison + click correlation + regression

In [ ]:
# Per-condition summary
rows = []
for ad, sub in test_eval.groupby(AD_COL):
    sub_valid = sub[['CosineSimilarity', 'EuclideanDistance', 'CLICKTHRU']].dropna()
    if len(sub_valid) < 3:
        corr_val, corr_p = np.nan, np.nan
    else:
        try:
            corr_val, corr_p = pearsonr(sub_valid['CosineSimilarity'], sub_valid['CLICKTHRU'])
        except Exception:
            corr_val, corr_p = np.nan, np.nan

    rows.append({
        AD_COL: ad,
        'n_test': len(sub),
        'EuclideanDistance_Mean': sub['EuclideanDistance'].mean(),
        'CosineSimilarity_Mean': sub['CosineSimilarity'].mean(),
        'Corr_Cosine_Click': corr_val,
        'Corr_Cosine_Click_p': corr_p
    })

table8 = pd.DataFrame(rows).sort_values(AD_COL).reset_index(drop=True)
table8.to_csv(os.path.join(OUTPUT_DIR, 'table8.csv'), index=False)
print('Saved:', os.path.join(OUTPUT_DIR, 'table8.csv'))
display(table8.head(50))

# Regression comparison on pooled TEST rows
reg_data = test_eval[['CLICKTHRU', 'EuclideanDistance', 'CosineSimilarity']].dropna().copy()

if len(reg_data) < 10:
    raise ValueError('Not enough valid TEST rows for regression comparison.')

# Model 1: CLICKTHRU ~ EuclideanDistance
X1 = sm.add_constant(reg_data[['EuclideanDistance']])
m1 = sm.OLS(reg_data['CLICKTHRU'], X1).fit()

# Model 2: CLICKTHRU ~ CosineSimilarity
X2 = sm.add_constant(reg_data[['CosineSimilarity']])
m2 = sm.OLS(reg_data['CLICKTHRU'], X2).fit()

reg_summary_lines = []
reg_summary_lines.append('Regression comparison on TEST rows')
reg_summary_lines.append('-----------------------------------')

coef1 = m1.params.get('EuclideanDistance', np.nan)
se1 = m1.bse.get('EuclideanDistance', np.nan)
p1 = m1.pvalues.get('EuclideanDistance', np.nan)
r21 = m1.rsquared
reg_summary_lines.append(f"Model A: CLICKTHRU ~ EuclideanDistance")
reg_summary_lines.append(f"  beta = {coef1:.6f}, SE = {se1:.6f}, p = {p1:.6g}, R^2 = {r21:.6f}")

coef2 = m2.params.get('CosineSimilarity', np.nan)
se2 = m2.bse.get('CosineSimilarity', np.nan)
p2 = m2.pvalues.get('CosineSimilarity', np.nan)
r22 = m2.rsquared
reg_summary_lines.append(f"Model B: CLICKTHRU ~ CosineSimilarity")
reg_summary_lines.append(f"  beta = {coef2:.6f}, SE = {se2:.6f}, p = {p2:.6g}, R^2 = {r22:.6f}")

reg_summary_text = '
'.join(reg_summary_lines)
print(reg_summary_text)

with open(os.path.join(OUTPUT_DIR, 'regression_summary.txt'), 'w') as f:
    f.write(reg_summary_text + '

')
    f.write('--- Full model summaries ---

')
    f.write(str(m1.summary()))
    f.write('

')
    f.write(str(m2.summary()))

print('Saved:', os.path.join(OUTPUT_DIR, 'regression_summary.txt'))

## 6) Table 9: Cross-group validation (vector stability)

Participants are split into Group 1 and Group 2 by `ResponseId`. Vectors are estimated independently in each group and compared by cosine similarity per ad condition.

In [ ]:
def derive_vectors_for_ids(full_df, include_ids, ad_col, trait_cols, outcome_col, min_n):
    subset = full_df[full_df[ID_COL].isin(include_ids)].copy()
    vec_df, skipped = fit_ad_signature_vectors(
        train_data=subset,
        ad_col=ad_col,
        trait_cols=trait_cols,
        outcome_col=outcome_col,
        min_n=min_n
    )
    return vec_df, skipped

all_ids = df[ID_COL].unique()
g1_ids, g2_ids = train_test_split(all_ids, test_size=0.5, random_state=RANDOM_SEED)

v1, s1 = derive_vectors_for_ids(df, g1_ids, AD_COL, TRAIT_COLS, SIGNATURE_OUTCOME, MIN_TRAIN_N_PER_AD)
v2, s2 = derive_vectors_for_ids(df, g2_ids, AD_COL, TRAIT_COLS, SIGNATURE_OUTCOME, MIN_TRAIN_N_PER_AD)

common_ads = sorted(set(v1[AD_COL]).intersection(set(v2[AD_COL])))
if len(common_ads) == 0:
    raise ValueError('No overlapping ad conditions with estimable vectors in Group1 and Group2.')

t9_rows = []
for ad in common_ads:
    r1 = v1.loc[v1[AD_COL] == ad].iloc[0]
    r2 = v2.loc[v2[AD_COL] == ad].iloc[0]

    vec1 = np.array([r1[c] for c in TRAIT_COLS], dtype=float)
    vec2 = np.array([r2[c] for c in TRAIT_COLS], dtype=float)

    n1 = np.linalg.norm(vec1)
    n2 = np.linalg.norm(vec2)
    cos12 = np.dot(vec1, vec2) / (n1 * n2) if (n1 > 0 and n2 > 0) else np.nan

    row = {
        AD_COL: ad,
        'Group1_Vector_Norm': n1,
        'Group2_Vector_Norm': n2,
        'Stability_Cosine_G1_vs_G2': cos12,
    }
    for c in TRAIT_COLS:
        row[f'G1_{c}'] = r1[c]
        row[f'G2_{c}'] = r2[c]
    t9_rows.append(row)

table9 = pd.DataFrame(t9_rows).sort_values(AD_COL).reset_index(drop=True)
table9.to_csv(os.path.join(OUTPUT_DIR, 'table9.csv'), index=False)
print('Saved:', os.path.join(OUTPUT_DIR, 'table9.csv'))
display(table9.head(50))

if len(s1) > 0:
    print('
Group 1 skipped ads:')
    display(s1)
if len(s2) > 0:
    print('
Group 2 skipped ads:')
    display(s2)

## 7) Appendix Figure A: 3D PCA projection of ad vectors

In [ ]:
vec_for_pca = vector_df[[AD_COL] + TRAIT_COLS].dropna().copy()

if len(vec_for_pca) >= 2:
    X = vec_for_pca[TRAIT_COLS].values
    Xs = StandardScaler().fit_transform(X)
    pca = PCA(n_components=3)
    Z = pca.fit_transform(Xs)

    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')

    ax.scatter(Z[:,0], Z[:,1], Z[:,2], s=70)
    for i, label in enumerate(vec_for_pca[AD_COL].astype(str).values):
        ax.text(Z[i,0], Z[i,1], Z[i,2], label, fontsize=8)

    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_zlabel('PC3')
    ax.set_title('3D PCA Projection of TRAIN-derived Ad Signature Vectors')

    pca_path = os.path.join(OUTPUT_DIR, 'figure_pca_3d_projection.png')
    plt.tight_layout()
    plt.savefig(pca_path, dpi=200)
    plt.show()
    print('Saved:', pca_path)
else:
    print('Not enough ad vectors for PCA plot.')

## 8) Appendix Figure B: Radar chart(s) of normalized ad vectors

In [ ]:
def radar_plot_single(ax, labels, values, title=''):
    angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
    values = values.tolist()
    values += values[:1]
    angles += angles[:1]

    ax.plot(angles, values, linewidth=1.5)
    ax.fill(angles, values, alpha=0.15)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels)
    ax.set_title(title, fontsize=9)

labels = ['Openness', 'Conscientiousness', 'Extraversion', 'Agreeableness', 'Neuroticism']
unit_cols = [f'{c}_unit' for c in TRAIT_COLS]
rad_df = vector_df[[AD_COL] + unit_cols].copy()

if RADAR_MODE == 'combined':
    fig = plt.figure(figsize=(10, 10))
    ax = plt.subplot(111, polar=True)

    angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
    angles += angles[:1]

    for _, r in rad_df.iterrows():
        vals = [r[c] for c in unit_cols]
        vals += vals[:1]
        ax.plot(angles, vals, linewidth=1.2, label=str(r[AD_COL]))

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels)
    ax.set_title('Normalized 5D Ad Signature Vectors (Combined Radar)')
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=7)

    radar_path = os.path.join(OUTPUT_DIR, 'figure_radar_combined.png')
    plt.tight_layout()
    plt.savefig(radar_path, dpi=200)
    plt.show()
    print('Saved:', radar_path)

elif RADAR_MODE == 'per_condition':
    saved = []
    for _, r in rad_df.iterrows():
        fig = plt.figure(figsize=(6, 6))
        ax = plt.subplot(111, polar=True)
        radar_plot_single(ax, labels, r[unit_cols].values, title=f"{r[AD_COL]} (normalized)")
        fpath = os.path.join(OUTPUT_DIR, f"figure_radar_{str(r[AD_COL]).replace('/', '_')}.png")
        plt.tight_layout()
        plt.savefig(fpath, dpi=200)
        plt.close(fig)
        saved.append(fpath)
    print(f'Saved {len(saved)} radar files.')

else:
    print("RADAR_MODE must be 'combined' or 'per_condition'.")

## 9) Appendix Figure C: Cosine angle visualization (example TEST row)

In [ ]:
example = test_eval[['CosineSimilarity', AD_COL] + TRAIT_COLS + [f'{c}_ad' for c in TRAIT_COLS]].dropna().head(1)

if len(example) == 0:
    print('No valid example row for cosine visualization.')
else:
    ex = example.iloc[0]
    p_vec = np.array([ex[c] for c in TRAIT_COLS], dtype=float)
    a_vec = np.array([ex[f'{c}_ad'] for c in TRAIT_COLS], dtype=float)
    ad_name = ex[AD_COL]

    cos_val = np.dot(p_vec, a_vec) / (np.linalg.norm(p_vec) * np.linalg.norm(a_vec))

    idx = np.arange(len(TRAIT_COLS))
    trait_labels = ['ZOPEN', 'ZCONSC', 'ZEXTRA', 'ZAGREE', 'ZNEURO']

    plt.figure(figsize=(9, 5))
    plt.plot(idx, p_vec, marker='o', label='Participant 5D vector')
    plt.plot(idx, a_vec, marker='s', label=f'Ad signature vector ({ad_name})')
    plt.xticks(idx, trait_labels)
    plt.axhline(0, color='gray', linewidth=0.8)
    plt.title(f'Cosine Angle Visualization | cosine = {cos_val:.4f}')
    plt.ylabel('Vector component value')
    plt.legend()
    plt.tight_layout()

    cos_path = os.path.join(OUTPUT_DIR, 'figure_cosine_angle_visualization.png')
    plt.savefig(cos_path, dpi=200)
    plt.show()

    print(f'Example AdCond: {ad_name}')
    print(f'Cosine similarity: {cos_val:.6f}')
    print('Saved:', cos_path)

## 10) Save quick notebooks summaries and preview output files

In [ ]:
# Save top-level summary tables to screen
print('
--- Table 7 preview ---')
display(table7.head(10))

print('
--- Table 8 preview ---')
display(table8.head(10))

print('
--- Table 9 preview ---')
display(table9.head(10))

print('
Output directory contents:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(' -', f)

## 11) Zip outputs and download

In [ ]:
zip_path = '/content/dissertation_extension_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(OUTPUT_DIR):
        full = os.path.join(OUTPUT_DIR, f)
        if os.path.isfile(full):
            zf.write(full, arcname=f)

print('Created zip:', zip_path)

try:
    from google.colab import files
    files.download(zip_path)
    print('Download triggered.')
except Exception as e:
    print('Auto-download unavailable in this environment. File is at:', zip_path)
    print('Details:', str(e))